# `drcs_activations` → `nasa-disasters-staging`

**Source** `s3://nasa-disasters/drcs_activations/<EVENT>/<SUB_PRODUCT>/` — raw, non-COG GeoTIFFs
staged for one activation, one product folder per run.

**Destination** `s3://nasa-disasters-staging/ProgramData/<PRODUCT>/Output/<stem>.tif`. The event is
**not** in the filename; it lives in the GeoTIFF tags (`ACTIVATION_EVENT`, `YEAR_MONTH`, `HAZARD`,
`LOCATION`, `SOURCE`, `PROCESSOR`) and in the S3 prefix. A source name that already carries the
event prefix has it stripped.

Three steps: **list** the folder → **convert** each file to a COG with the tags embedded → **upload**.
Fixed choices, on purpose: native CRS and ZSTD. The one per-run knob that matters is `NODATA`.

> For COGs already under the sensor-first `drcs_activations_new/<Sensor>/<product>/` tree use
> [`drcs_new_transfer.ipynb`](drcs_new_transfer.ipynb).

In [ ]:
# ---- INPUTS ----
EVENT_NAME = '202604_Typhoon_Sinlaku'    # YYYYMM_Hazard_Location
SOURCE = 'NASA'                          # data origin, baked into every COG as its SOURCE tag
SUB_PRODUCT = 'Blackmarble'              # folder under the event holding this run's files
PRODUCT = 'Blackmarble'                  # destination folder: ProgramData/<PRODUCT>/Output

SOURCE_BUCKET = 'nasa-disasters'
SOURCE_ROOT = 'drcs_activations'
DESTINATION_BUCKET = 'nasa-disasters-staging'
DESTINATION_TEMPLATE = 'ProgramData/{product}/Output'

# ---- NODATA: three settings, told apart by TYPE (the cog_utils.convert_to_cog contract) ----
#   NODATA = None     inherit the nodata tag the source already carries; if it has none,
#                     auto-detect from dtype (int/float -> -9999; bare 8-bit -> nothing declared)
#   NODATA = False    declare NO nodata, and STRIP any tag the source carries. This is the
#                     8-bit imagery case (RGB composites): every value 0-255 is a real sample,
#                     and an inherited 0 masks legitimately dark pixels
#   NODATA = -9999    declare exactly this value (a vendor-reserved fill). Never 0 for a
#                     float product where 0 is real data (displacement, dB backscatter)
# True is rejected outright (a bool is an int in Python; it would be declared as 1).
NODATA = None
OVERWRITE = False          # True to replace an object already in the destination
COMPRESSION_LEVEL = 9      # ZSTD level: 1 = fast/larger ... 22 = slow/smallest

In [ ]:
from shared_utils import PROCESSOR_STRING
ACTIVATION_METADATA = {
    "ACTIVATION_EVENT": EVENT_NAME,
    "SOURCE": SOURCE,
    "PROCESSOR": PROCESSOR_STRING,
}

## Step 1: List the source folder

In [ ]:
import os
import boto3

s3 = boto3.client('s3')   # ambient credentials; on the hub the pod already assumes disasters-prod
prefix = f'{SOURCE_ROOT}/{EVENT_NAME}/{SUB_PRODUCT}/'

found = []   # (key, size_bytes)
for page in s3.get_paginator('list_objects_v2').paginate(Bucket=SOURCE_BUCKET, Prefix=prefix):
    for obj in page.get('Contents', []):
        if obj['Key'].lower().endswith('.tif'):   # .tif / .TIF -- vendor deliveries vary
            found.append((obj['Key'], obj['Size']))
found.sort()

print(f"s3://{SOURCE_BUCKET}/{prefix}")
for key, size in found:
    print(f"    {os.path.basename(key):<80} {size / 1024**2:8.1f} MB")
print(f"\nFound {len(found)} .tif file(s)")
if not found:
    print(f"Nothing here. Check from a terminal:  aws s3 ls s3://{SOURCE_BUCKET}/{prefix}")

## Step 2: Plan the output names

In [ ]:
from shared_utils.file_naming import strip_event_prefix, create_nisar_filename

# The event stays OUT of the filename (it lives in the tags + the S3 prefix) and the
# stem is normalised to the repo convention. create_nisar_filename is the shared
# builder that also keeps BOTH dates of a pair product (NISAR GUNW); for every other
# name it is identical to create_output_filename, and it is idempotent.
dest_prefix = DESTINATION_TEMPLATE.format(product=PRODUCT)
plan = []
for key, size in found:
    filename = os.path.basename(key)
    new_name = create_nisar_filename(strip_event_prefix(filename, EVENT_NAME), '')
    plan.append({'source': key, 'new_name': new_name,
                 'dest_prefix': dest_prefix, 'dest': f"{dest_prefix}/{new_name}"})
    print(f"  {filename}\n    -> s3://{DESTINATION_BUCKET}/{dest_prefix}/{new_name}")

# main_processor writes its scratch COG to /tmp/cog_<new_name>, so two plan items
# with the same output name would overwrite each other under the thread pool.
_dupes = {p['new_name'] for p in plan if sum(q['new_name'] == p['new_name'] for q in plan) > 1}
if _dupes:
    raise RuntimeError(f"Duplicate output name(s) in this run: {sorted(_dupes)}")
print(f"\n{len(plan)} file(s) to process")

## Step 3: Convert to COG with tags, upload

In [ ]:
from shared_utils.main_processor import convert_to_cog
from shared_utils.parallel import map_threaded
from shared_utils.s3_operations import can_write_to_bucket

# Prove a real PutObject under every destination prefix BEFORE converting anything:
# grants on this bucket are per-prefix, and a read-only identity still passes head_bucket.
for dest_prefix in sorted({p['dest_prefix'] for p in plan}):
    ok, detail = can_write_to_bucket(s3, DESTINATION_BUCKET, dest_prefix)
    if not ok:
        raise RuntimeError(f"Cannot write to s3://{DESTINATION_BUCKET}/{dest_prefix}/ -- {detail}")


def _process(item):
    # download (or /vsis3 stream) -> COG with the activation tags embedded -> upload.
    # Raises FileExistsError when the destination exists and OVERWRITE is False.
    convert_to_cog(
        item['source'], SOURCE_BUCKET, item['new_name'],
        DESTINATION_BUCKET, item['dest_prefix'], s3,
        manual_nodata=NODATA, overwrite=OVERWRITE, target_crs=None,
        metadata=ACTIVATION_METADATA, compression='ZSTD', compression_level=COMPRESSION_LEVEL,
    )
    return 'success'


outcomes = map_threaded(_process, plan, max_workers=4, desc="Publish to staging")

results = {'success': [], 'skipped': [], 'failed': []}
for item, out in zip(plan, outcomes):
    if isinstance(out, FileExistsError):
        results['skipped'].append((item, out))
    elif isinstance(out, Exception):
        results['failed'].append((item, out))
    else:
        results['success'].append((item, out))

print(f"\nsuccess {len(results['success'])}   "
      f"skipped (already in destination) {len(results['skipped'])}   "
      f"failed {len(results['failed'])}")
for item, err in results['failed']:
    print(f"  FAILED {item['source']}: {err}")

## Step 4: Verify the tags on one uploaded COG

In [ ]:
import rasterio

# The event is not in the filename, so these tags are the only record of it.
REQUIRED_TAGS = ('ACTIVATION_EVENT', 'YEAR_MONTH', 'HAZARD', 'LOCATION', 'SOURCE', 'PROCESSOR')

if not results['success']:
    print("Nothing was uploaded this run; nothing to verify.")
else:
    item = results['success'][0][0]
    local = f"/tmp/verify_{item['new_name']}"
    try:
        s3.download_file(DESTINATION_BUCKET, item['dest'], local)
        with rasterio.open(local) as src:
            tags = src.tags()
        for t in REQUIRED_TAGS:
            print(f"  {t:<17} {tags.get(t, '!! MISSING')}")
        missing = [t for t in REQUIRED_TAGS if t not in tags]
        if missing:
            raise RuntimeError(f"s3://{DESTINATION_BUCKET}/{item['dest']} is missing {missing}")
        print(f"\nAll activation tags present on s3://{DESTINATION_BUCKET}/{item['dest']}")
    finally:
        if os.path.exists(local):
            os.remove(local)